In [ ]:
import google.generativeai as genai
import os

# ใส่ Key ของคุณตรงนี้
# genai.configure(api_key="")

print("List of available models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

List of available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robot

In [6]:
# ---------------------------------------------------------
# 1. DATABASE CONNECTION
# ---------------------------------------------------------
DB_USER = os.getenv("DATABASE_USER", "hospital_user")
DB_PASS = os.getenv("DATABASE_PASSWORD", "hospital_pass")
DB_HOST = os.getenv("DATABASE_HOST", "hospital_db")
DB_NAME = os.getenv("DATABASE_NAME", "hospital_db")

# เชื่อมต่อ Database
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}")

# ---------------------------------------------------------
# 2. QUERY DATA & GEOMETRY
# ---------------------------------------------------------
# ดึงข้อมูล Shapefile และ Data Table มารวมกัน

In [ ]:
# ---------------------------------------------------------
# 2. QUERY DATA & GEOMETRY
# ---------------------------------------------------------
# ดึงข้อมูล Shapefile และ Data Table มารวมกัน
query = """
    SELECT 
        g."ProvinceKey",
        g."ADM1_TH" AS province_name_th,
        g.geometry,
        a.*
    FROM "Readiness_geo" g
    JOIN "Readiness" a
    ON g."ProvinceKey" = a."ProvinceKey";
"""
gdf = gpd.read_postgis(query, engine, geom_col="geometry")

# ลบคอลัมน์ที่ชื่อซ้ำกัน (ถ้ามี)
gdf = gdf.loc[:, ~gdf.columns.duplicated()]

# ---------------------------------------------------------
# 3. IDENTIFY DYNAMIC COLUMNS (ระบุคอลัมน์อัตโนมัติ)
# ---------------------------------------------------------
# ดึงรายชื่อคอลัมน์ทั้งหมดที่ขึ้นต้นด้วย doctors_ และ equip_ เพื่อใช้ในการคำนวณและส่งไปหน้าเว็บ
doctor_cols = [c for c in gdf.columns if c.startswith('doctors_')]
equip_cols = [c for c in gdf.columns if c.startswith('equip_')]

# การจับคู่โรคและ Supply สำหรับคำนวณ Penalty
disease_mapping = {
    'disease_dm': { 
        'burden': 'disease_dm', 
        'supply': ['doctors_pediatric_endocrine', 'doctors_ophthalmology', 'doctors_vascular_surgery']
    },
    'disease_heart': { 
        'burden': 'disease_heart', 
        'supply': ['doctors_pediatric_cardiology', 'equip_mri', 'equip_ct_scanner']
    }
}

# ---------------------------------------------------------
# 4. CALCULATION: R_final Model
# ---------------------------------------------------------

# 4.1 เตรียมตัวหาร (ประชากรผู้สูงอายุ) - ป้องกันการหารด้วย 0
elderly_pop = gdf['elderly_population'].replace(0, 1)

# 4.2 คำนวณค่า Raw Per Capita (หารด้วยผู้สูงอายุ)
# X1: Personnel (รวมหมอทุกประเภท)
gdf['X1_raw_per_capita'] = gdf[doctor_cols].sum(axis=1) / elderly_pop

# X2: Equipment (รวมอุปกรณ์ทุกประเภท)
gdf['X2_raw_per_capita'] = gdf[equip_cols].sum(axis=1) / elderly_pop

# X3: Insurance (บัตรทอง)
gdf['X3_raw_per_capita'] = gdf['insurance_uc_scheme'] / elderly_pop

# X4: Load (ผู้ป่วยใน) -> ยิ่งเยอะยิ่งแย่ (เดี๋ยว Invert ตอน Z-score)
gdf['X4_raw_per_capita'] = gdf['ipd_avg_inpatients_per_day'] / elderly_pop